In [ ]:
# 1. Load và phân tách dữ liệu C-MAPSS
import sys
import pprint
import numpy as np
import pandas as pd
from pathlib import Path

from demo.core.data import (
    SplitConfig,
    prepare_cmapss_split,
)

config = SplitConfig(
    data_path="demo/data/train_FD002.txt",
    train_ratio=0.70,
    observation_cycle_min=130,
    observation_cycle_max=135,
    rare_fault_probability=0.20,
    rare_fault_max_duration=10,
    rare_fault_start_cycle_min=60,
    rare_fault_magnitude_min=0.03,
    rare_fault_magnitude_max=0.10,
    rare_fault_types=("plateau", "drop", "drift"),
    test_fraction_min=0.10,
    test_fraction_max=0.30,
    random_seed=42,
)

# Chạy hàm tạo dữ liệu
data = prepare_cmapss_split(config)

# Trích xuất trực tiếp toàn bộ các phần tử được return ra từng biến riêng biệt
df                      = data["df"]
engine_data             = data["engine_data"]
op_data                 = data["op_data"]
train_data              = data["train_data"]
train_op_data           = data["train_op_data"]
train_ids               = data["train_ids"]
train_metadata          = data["train_metadata"]
test_data               = data["test_data"]
test_op_data            = data["test_op_data"]
test_ids                = data["test_ids"]
test_observation_points = data["test_observation_points"]
sensors                 = data["sensors"]
op_settings             = data["op_settings"]

print("Đã nạp và giải nén toàn bộ biến thành công!")
print(f"- Số lượng cảm biến (sensors): {sensors}")
print(f"- Số lượng cài đặt vận hành (op_settings): {op_settings}")
print(f"- Tổng số động cơ Train: {len(train_ids)}")
print(f"- Tổng số động cơ Test : {len(test_ids)}")

In [ ]:
# 2. Kiểm tra dữ liệu thuần của tập TRAIN
# Chọn 1 ID động cơ bất kỳ trong tập train để xem
train_id = train_ids[0]  # Ví dụ engine 1

print(f"==================== ENGINE TRAIN #{train_id} ====================")
print("\n--- [train_data] Mảng cảm biến đã cắt (Shape, 5 dòng đầu) ---")
print("Shape:", train_data[train_id].shape)
print(train_data[train_id][:5])

print("\n--- [train_op_data] Mảng thiết lập vận hành (Shape, 5 dòng đầu) ---")
print("Shape:", train_op_data[train_id].shape)
print(train_op_data[train_id][:5])

print("\n--- [train_metadata] Thông tin chi tiết ---")
pprint.pprint(train_metadata[train_id])

In [ ]:
# 3. Lọc và xuất dữ liệu của các động cơ có TIÊM LỖI HIẾM trong tập Train
fault_engines = [eid for eid in train_ids if train_metadata[eid]["has_rare_fault"]]
print(f"Số lượng động cơ bị tiêm lỗi hiếm: {len(fault_engines)} / {len(train_ids)}")
print(f"Danh sách ID động cơ có lỗi: {fault_engines[:10]}...")

# Xem chi tiết động cơ có lỗi đầu tiên
if fault_engines:
    sample_fault_id = fault_engines[0]
    print(f"\nMetadata của động cơ có lỗi #{sample_fault_id}:")
    pprint.pprint(train_metadata[sample_fault_id])

In [ ]:
# 4. Kiểm tra dữ liệu thuần của tập TEST (Dữ liệu quan sát ban đầu)
test_id = test_ids[0]

print(f"==================== ENGINE TEST #{test_id} ====================")
print(f"Điểm quan sát ban đầu (test_observation_points): {test_observation_points[test_id]} chu kỳ")

print("\n--- [test_data] Mảng cảm biến ban đầu (Shape, 5 dòng đầu) ---")
print("Shape:", test_data[test_id].shape)
print(test_data[test_id][:5])

print("\n--- [test_op_data] Mảng vận hành ban đầu (Shape, 5 dòng đầu) ---")
print("Shape:", test_op_data[test_id].shape)
print(test_op_data[test_id][:5])

In [ ]:
# 5. Kiểm tra dữ liệu GỐC ĐẦY ĐỦ (Ground Truth: engine_data, op_data, df)
print(f"Tổng số động cơ trong toàn bộ dataset: {len(engine_data)}")

print(f"\n--- [engine_data] Chuỗi cảm biến đầy đủ từ cycle 1 đến hỏng của engine #{test_id} ---")
print("Full Shape:", engine_data[test_id].shape)
print(f"(Test chỉ quan sát {test_data[test_id].shape[0]} chu kỳ đầu, còn lại {engine_data[test_id].shape[0] - test_data[test_id].shape[0]} chu kỳ bị ẩn)")

print("\n--- [df] Bảng DataFrame C-MAPSS gốc đã lọc cột (5 dòng đầu) ---")
display(df.head())